# NB: 想定めぐ指数 — 実測 MAE 最小化チューニング

**目的**: 過去走ブレンド・条件重み・キャリブレーションを train/valid/test 分割で最適化し、実測 `megu_index` との MAE を最小化する（**1点=0.1秒は不変**）。

## 分割方針（時系列・レース単位）

| split | 期間 |
|-------|------|
| train | 2024年まで |
| valid | 2025-01-01 〜 2025-06-30 |
| test  | 2025-07-01 以降 |

グリッドサーチは train のサブサンプルで実施し、valid でモデル選択、test で最終報告。

## 参照
- `src/pipeline/megu_index/optimize_predict.py`
- `src/pipeline/megu_index/predict_params.py`
- `config/megu_predict_params.json`

## 0. セットアップ

In [ ]:
import json
import sys
import os
from dataclasses import asdict
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sqlalchemy import create_engine

REPO_ROOT = Path("../../").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

from dotenv import load_dotenv
load_dotenv(REPO_ROOT / ".env.stg", override=False)
load_dotenv(REPO_ROOT / ".env", override=False)

from src.pipeline.megu_index.optimize_condition_weights import fetch_eval_frame
from src.pipeline.megu_index.optimize_predict import (
    evaluate_params,
    grid_search_predict,
    refine_bias,
    load_transfer_map,
    load_beta_weight,
    split_temporal,
    metrics_to_dict,
    _subsample_df,
)
from src.pipeline.megu_index.predict_params import PredictParams, save_predict_params

OUT_DIR = REPO_ROOT / "notebooks/modeling/_run_output/megu_predict_opt"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATABASE_URL = os.environ.get("DATABASE_URL")
assert DATABASE_URL, "DATABASE_URL が未設定です (.env.stg)"
engine = create_engine(DATABASE_URL)
print("REPO_ROOT:", REPO_ROOT)
print("OUT_DIR:", OUT_DIR)

## 1. データロードと train/valid/test 分割

In [ ]:
YEAR_START, YEAR_END = 2024, 2025
TRAIN_END = "2024-12-31"
VALID_END = "2025-06-30"
TRAIN_SAMPLE = 8000

df_all = fetch_eval_frame(engine, year_start=YEAR_START, year_end=YEAR_END)
df_train, df_valid, df_test = split_temporal(df_all, train_end=TRAIN_END, valid_end=VALID_END)

split_summary = pd.DataFrame([
    {"split": "train", "pairs": len(df_train), "races": df_train["race_id"].nunique(),
     "date_min": df_train["race_date"].min(), "date_max": df_train["race_date"].max()},
    {"split": "valid", "pairs": len(df_valid), "races": df_valid["race_id"].nunique(),
     "date_min": df_valid["race_date"].min(), "date_max": df_valid["race_date"].max()},
    {"split": "test", "pairs": len(df_test), "races": df_test["race_id"].nunique(),
     "date_min": df_test["race_date"].min(), "date_max": df_test["race_date"].max()},
    {"split": "all", "pairs": len(df_all), "races": df_all["race_id"].nunique(),
     "date_min": df_all["race_date"].min(), "date_max": df_all["race_date"].max()},
])
split_summary

## 2. ベースライン（デフォルトパラメータ）

In [ ]:
transfer_map = load_transfer_map(engine)
beta_weight = load_beta_weight(engine)
default_params = PredictParams()

baseline_rows = []
for name, part in [("train", df_train), ("valid", df_valid), ("test", df_test)]:
    m = evaluate_params(part, default_params, transfer_map=transfer_map, beta_weight=beta_weight)
    baseline_rows.append(metrics_to_dict(m, name))
baseline_df = pd.DataFrame(baseline_rows)
baseline_df

## 3. train でグリッドサーチ → valid で選択

In [ ]:
df_train_fit = _subsample_df(df_train, TRAIN_SAMPLE)
print(f"grid fit sample: {len(df_train_fit):,} pairs / train {len(df_train):,}")

best_params, grid_df, train_m = grid_search_predict(
    df_train_fit,
    transfer_map=transfer_map,
    beta_weight=beta_weight,
)
best_params, refine_train_m = refine_bias(
    df_train_fit,
    best_params,
    transfer_map=transfer_map,
    beta_weight=beta_weight,
)
print("grid best tuning:", asdict(best_params.tuning))
print("condition_weights:", best_params.condition_weights)
print("history_weights:", best_params.history_weights)
grid_df.head(10)

In [ ]:
valid_m = evaluate_params(df_valid, best_params, transfer_map=transfer_map, beta_weight=beta_weight)
train_full_m = evaluate_params(df_train, best_params, transfer_map=transfer_map, beta_weight=beta_weight)
opt_rows = [
    metrics_to_dict(refine_train_m, "train_sample"),
    metrics_to_dict(train_full_m, "train_full"),
    metrics_to_dict(valid_m, "valid"),
]
pd.DataFrame(opt_rows)

## 4. test セット最終評価

In [ ]:
test_m = evaluate_params(df_test, best_params, transfer_map=transfer_map, beta_weight=beta_weight)
all_m = evaluate_params(df_all, best_params, transfer_map=transfer_map, beta_weight=beta_weight)

final_metrics = pd.DataFrame([
  *baseline_rows,
  metrics_to_dict(train_full_m, "optimized_train"),
  metrics_to_dict(valid_m, "optimized_valid"),
  metrics_to_dict(test_m, "optimized_test"),
  metrics_to_dict(all_m, "optimized_all"),
])
final_metrics["mae_improve_vs_baseline"] = final_metrics["mae"] - baseline_df["mae"].iloc[0]
final_metrics

## 5. 予測 vs 実測 散布図（test）

In [ ]:
from src.pipeline.megu_index.optimize_predict import predict_row

preds, actuals = [], []
for _, row in df_test.iterrows():
    p = predict_row(row, best_params, transfer_map=transfer_map, beta_weight=beta_weight)
    if p is None:
        continue
    preds.append(p)
    actuals.append(float(row["actual_megu"]))

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(actuals, preds, alpha=0.15, s=8)
lo = min(min(actuals), min(preds))
hi = max(max(actuals), max(preds))
ax.plot([lo, hi], [lo, hi], "r--", lw=1)
ax.set_xlabel("actual megu")
ax.set_ylabel("predicted megu")
ax.set_title(f"test set (MAE={test_m.mae:.2f}, n={len(preds):,})")
fig.tight_layout()
fig.savefig(OUT_DIR / "scatter_test.png", dpi=120)
plt.show()

## 6. パラメータ保存と成果物出力

In [ ]:
best_params.meta = {
    "model_version": "v2",
    "year_start": YEAR_START,
    "year_end": YEAR_END,
    "split": {
        "train_end": TRAIN_END,
        "valid_end": VALID_END,
        "train_sample": TRAIN_SAMPLE,
    },
    "baseline": baseline_df.to_dict(orient="records"),
    "optimized": final_metrics.to_dict(orient="records"),
    "note": "1点=0.1秒は不変。ability_bias_sec は秒単位キャリブレーション。",
}

config_path = save_predict_params(best_params)
grid_df.to_csv(OUT_DIR / "grid_results.csv", index=False)
final_metrics.to_csv(OUT_DIR / "metrics_summary.csv", index=False)
with open(OUT_DIR / "metrics_summary.json", "w", encoding="utf-8") as f:
    json.dump({
        "config_path": str(config_path),
        "best_params": {
            "tuning": asdict(best_params.tuning),
            "condition_weights": best_params.condition_weights,
            "history_weights": best_params.history_weights,
        },
        "metrics": final_metrics.to_dict(orient="records"),
    }, f, ensure_ascii=False, indent=2)

print("saved config:", config_path)
print("artifacts:", sorted(p.name for p in OUT_DIR.iterdir()))